In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retail_lakehouse")

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show()

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.retail_lakehouse.landing")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/retail_lakehouse/landing/retail_day_01"))

In [0]:
landing = "/Volumes/workspace/retail_lakehouse/landing/retail_day_01"
schema  = "workspace.retail_lakehouse"

# sanity check: can Spark see the files?
display(dbutils.fs.ls(landing))

In [0]:
customers_df = (
    spark.read
    .option("header", "true")       # first row is column names
    .option("inferSchema", "true")  # let Spark guess types from the data
    .csv(f"{landing}/customers.csv")
)

customers_df.printSchema()   # look at the columns + guessed types
customers_df.show(5)

In [0]:
from pyspark.sql import functions as F

customers_df = (
    customers_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("customers.csv"))
)

In [0]:
(
    customers_df.write
    .format("delta")
    .mode("overwrite")              # replace the table each run (fine for Bronze rebuilds)
    .saveAsTable(f"{schema}.bronze_customers")
)

In [0]:
for name in ["products", "orders"]:
    df = (spark.read
          .option("header", "true").option("inferSchema", "true")
          .csv(f"{landing}/{name}.csv")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(f"{name}.csv")))
    (df.write.format("delta").mode("overwrite")
       .saveAsTable(f"{schema}.bronze_{name}"))
    print(f"wrote bronze_{name}: {df.count()} rows")

In [0]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

In [0]:
customers_df = customers_df.dropDuplicates(["customer_id"])

In [0]:
customers_df.dtypes

In [0]:
products_df = spark.read.table(f"{schema}.bronze_products")

In [0]:
# Assuming your DataFrame is named products_df
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Define the columns that define a duplicate (e.g., product_id)
# If you want to check the entire row, use products_df.columns instead
duplicate_columns = ["product_id"] 

# 2. Define the window. An orderBy is required for row_number()
# You can order by a timestamp, an ID, or literal(1) if order doesn't matter
window_spec = Window.partitionBy(duplicate_columns).orderBy(F.lit(1))

# 3. Create the boolean mask column (True for duplicates, False for unique/first occurrence)
products_df_with_mask = products_df.withColumn(
    "is_duplicate", 
    F.row_number().over(window_spec) > 1
)

# Show the results
products_df_with_mask.show()


In [0]:
products_df = products_df.dropDuplicates(["product_id"])

In [0]:
from pyspark.sql.functions import col, count, when

products_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_df.columns
]).show()

In [0]:
products_df.dtypes

In [0]:
orders_df = spark.read.table(f"{schema}.bronze_orders")

In [0]:
# Assuming your DataFrame is named products_df
from pyspark.sql import Window
from pyspark.sql import functions as F

# 1. Define the columns that define a duplicate (e.g., product_id)
# If you want to check the entire row, use products_df.columns instead
duplicate_columns = ["order_id"] 

# 2. Define the window. An orderBy is required for row_number()
# You can order by a timestamp, an ID, or literal(1) if order doesn't matter
window_spec = Window.partitionBy(duplicate_columns).orderBy(F.lit(1))

# 3. Create the boolean mask column (True for duplicates, False for unique/first occurrence)
orders_df_with_mask = orders_df.withColumn(
    "is_duplicate", 
    F.row_number().over(window_spec) > 1
)

# Show the results
orders_df_with_mask.show()


In [0]:
from pyspark.sql.functions import col, count, when

orders_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in orders_df.columns
]).show()

In [0]:
orders_df.dtypes

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, col
orders_df = (
    orders_df
    .withColumn("order_date", to_date(col("order_ts")))       # timestamp -> just the date
)

In [0]:
dfs_to_write = {
    "silver_customers": customers_df,
    "silver_products":  products_df,
    "silver_orders":    orders_df,
}

for table_name, df in dfs_to_write.items():
    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(f"workspace.retail_lakehouse.{table_name}"))
    print(f"wrote {table_name}: {df.count()} rows")

In [0]:
orders_enriched_df = (
    orders_df
    .join(customers_df, on="customer_id", how="left")
    .join(products_df, on="product_id", how="left")
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        orders_df["product_id"],
        orders_df["quantity"],
        orders_df["amount"],
        orders_df["order_ts"],
        orders_df["order_date"],
        customers_df["city"],
        customers_df["state"],
        customers_df["age"],
        customers_df["gender"],
        products_df["category"],
        products_df["brand"],
        products_df["price"],
    )
)

In [0]:
(orders_enriched_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.silver_orders_enriched"))

print(f"wrote silver_orders_enriched: {orders_enriched_df.count()} rows")

In [0]:
spark.sql("SHOW TABLES IN workspace.retail_lakehouse").show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# read from the table, not memory — same habit as the join step
orders_enriched_df = spark.table("workspace.retail_lakehouse.silver_orders_enriched")

w = Window.partitionBy("customer_id").orderBy(col("order_ts").desc())

latest_purchase_df = (
    orders_enriched_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

(latest_purchase_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.silver_customer_latest_purchase"))

print(f"wrote silver_customer_latest_purchase: {latest_purchase_df.count()} rows")

In [0]:
spark.sql("SHOW TABLES IN workspace.retail_lakehouse").show()

In [0]:
for t in ["silver_customers", "silver_products", "silver_orders",
          "silver_orders_enriched", "silver_customer_latest_purchase"]:
    n = spark.table(f"workspace.retail_lakehouse.{t}").count()
    print(f"{t}: {n}")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/retail_lakehouse/landing"))

In [0]:
base = "/Volumes/workspace/retail_lakehouse/landing"

display(dbutils.fs.ls(base))            # should now show only retail_day_01/ and retail_day_02/
display(dbutils.fs.ls(f"{base}/retail_day_02"))   # should show the 3 csvs

In [0]:
from pyspark.sql import functions as F

day2_path = "/Volumes/workspace/retail_lakehouse/landing/retail_day_02"  # adjust after the ls check

bronze_day2_orders = (
    spark.read
    .option("header", "true").option("inferSchema", "true")
    .csv(f"{day2_path}/orders.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("orders.csv"))
)

(bronze_day2_orders.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.retail_lakehouse.bronze_orders"))

print(f"appended {bronze_day2_orders.count()} rows to bronze_orders")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import to_timestamp, to_date, col

day2_orders_clean = (
    bronze_day2_orders
    .dropDuplicates(["order_id"])
    .dropna(subset=["order_id", "customer_id", "product_id", "amount"])
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("amount", col("amount").cast("double"))
    .withColumn("order_ts", to_timestamp(col("order_ts")))
    .withColumn("order_date", to_date(col("order_ts")))
)

silver_orders_tbl = DeltaTable.forName(spark, "workspace.retail_lakehouse.silver_orders")

(silver_orders_tbl.alias("target")
    .merge(day2_orders_clean.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

print(spark.table("workspace.retail_lakehouse.silver_orders").count())  # expect 600

In [0]:
from pyspark.sql import functions as F

tracked_cols = ["city", "state"]

silver_customers_scd = (
    spark.table("workspace.retail_lakehouse.silver_customers")
    .withColumn("effective_from", F.lit("2026-01-01").cast("date"))
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
    .withColumn("_scd_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in tracked_cols]), 256))
)

(silver_customers_scd.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")   # required: you're adding new columns, not just new rows
    .saveAsTable("workspace.retail_lakehouse.silver_customers"))

spark.table("workspace.retail_lakehouse.silver_customers").printSchema()

In [0]:
day2_path = "/Volumes/workspace/retail_lakehouse/landing/retail_day_02"

day2_customers = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(f"{day2_path}/customers.csv")
    .dropDuplicates(["customer_id"])
    .withColumn("_scd_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in tracked_cols]), 256))
)

In [0]:
from delta.tables import DeltaTable

customers_tbl = DeltaTable.forName(spark, "workspace.retail_lakehouse.silver_customers")
current_df = customers_tbl.toDF().filter("is_current = true")

# changed customers: hash differs from their current row -> force-insert a new version
changed = (
    day2_customers.alias("s")
    .join(current_df.alias("t"), "customer_id")
    .where("t._scd_hash != s._scd_hash")
    .select("s.*")
    .withColumn("mergeKey", F.lit(None).cast("string"))   # NULL never matches -> always goes to INSERT
)

# every source row, keyed by real customer_id -> used to close old rows / catch brand-new customers
to_match = day2_customers.withColumn("mergeKey", F.col("customer_id"))

staged = changed.unionByName(to_match)

(customers_tbl.alias("t")
    .merge(staged.alias("s"), "t.customer_id = s.mergeKey")
    .whenMatchedUpdate(
        condition="t.is_current = true AND t._scd_hash != s._scd_hash",
        set={"is_current": "false", "effective_to": "CAST('2026-01-02' AS DATE)"}
    )
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id", "name": "s.name", "city": "s.city",
        "state": "s.state", "age": "s.age", "gender": "s.gender",
        "signup_date": "s.signup_date",
        "effective_from": "CAST('2026-01-02' AS DATE)",
        "effective_to": "CAST(NULL AS DATE)",
        "is_current": "true", "_scd_hash": "s._scd_hash"
    })
    .execute())

In [0]:
t = spark.table("workspace.retail_lakehouse.silver_customers")
print(t.count())                                  # expect 203 (200 + 3 new versions)
t.filter("is_current = false").show()              # expect exactly the 3 closed-out old rows
t.filter("customer_id = 'C0105'").show(truncate=False)  # should show 2 rows: old Bangalore (closed), new Delhi (current)

In [0]:
spark.sql("""
    UPDATE workspace.retail_lakehouse.silver_customers
    SET _ingested_at = current_timestamp(), _source_file = 'customers.csv'
    WHERE _ingested_at IS NULL
""")

# verify: should be 0
spark.table("workspace.retail_lakehouse.silver_customers").filter("_ingested_at IS NULL").count()

In [0]:
silver_orders    = spark.table("workspace.retail_lakehouse.silver_orders")
silver_customers = spark.table("workspace.retail_lakehouse.silver_customers").filter("is_current = true")
silver_products  = spark.table("workspace.retail_lakehouse.silver_products")

orders_enriched_df = (
    silver_orders
    .join(silver_customers, on="customer_id", how="left")
    .join(silver_products, on="product_id", how="left")
    .select(
        silver_orders["order_id"], silver_orders["customer_id"], silver_orders["product_id"],
        silver_orders["quantity"], silver_orders["amount"],
        silver_orders["order_ts"], silver_orders["order_date"],
        silver_customers["city"], silver_customers["state"],
        silver_customers["age"], silver_customers["gender"],
        silver_products["category"], silver_products["brand"], silver_products["price"],
    )
)

(orders_enriched_df.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.silver_orders_enriched"))

print(orders_enriched_df.count())   # expect 600

In [0]:
from pyspark.sql import functions as F

orders_enriched = spark.table("workspace.retail_lakehouse.silver_orders_enriched")

gold_revenue_by_month = (
    orders_enriched
    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("year_month")
    .agg(
        F.sum("amount").alias("total_revenue"),
        F.count("order_id").alias("num_orders")
    )
    .orderBy("year_month")
)

(gold_revenue_by_month.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_revenue_by_month"))

gold_revenue_by_month.show()

In [0]:
from pyspark.sql import functions as F

orders_enriched = spark.table("workspace.retail_lakehouse.silver_orders_enriched")

In [0]:
gold_revenue_by_state = (
    orders_enriched
    .groupBy("state")
    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))
    .orderBy(F.desc("total_revenue"))
)
(gold_revenue_by_state.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_revenue_by_state"))

In [0]:
gold_revenue_by_category = (
    orders_enriched
    .groupBy("category")
    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))
    .orderBy(F.desc("total_revenue"))
)
(gold_revenue_by_category.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_revenue_by_category"))

In [0]:
gold_top_products = (
    orders_enriched
    .groupBy("product_id", "category", "brand")
    .agg(F.sum("amount").alias("total_revenue"), F.count("order_id").alias("num_orders"))
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)
(gold_top_products.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_top_products"))

In [0]:
gold_avg_basket_value = orders_enriched.agg(
    F.avg("amount").alias("avg_order_value"),
    F.count("order_id").alias("total_orders")
)
(gold_avg_basket_value.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_avg_basket_value"))

In [0]:
gold_top_customers = (
    orders_enriched
    .groupBy("customer_id", "city", "state")
    .agg(F.sum("amount").alias("total_spent"), F.count("order_id").alias("num_orders"))
    .orderBy(F.desc("total_spent"))
    .limit(10)
)
(gold_top_customers.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_top_customers"))

In [0]:
gold_repeat_customers = (
    orders_enriched
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("num_orders"),
        F.sum("amount").alias("total_spent")
    )
    .filter(F.col("num_orders") > 1)
    .orderBy(F.desc("num_orders"))
)
(gold_repeat_customers.write.format("delta").mode("overwrite")
    .saveAsTable("workspace.retail_lakehouse.gold_repeat_customers"))

In [0]:
for t in ["gold_revenue_by_month", "gold_revenue_by_state", "gold_revenue_by_category",
          "gold_top_products", "gold_top_customers", "gold_avg_basket_value", "gold_repeat_customers"]:
    n = spark.table(f"workspace.retail_lakehouse.{t}").count()
    print(f"{t}: {n} rows")